In [1]:
import sys
import os
project_root = os.path.abspath('..')
sys.path.append(project_root)
from src import AdversarialObjectDetection, test_noise_defense_with_iou
from src import DenoisingAE
from pathlib import Path
import numpy as np

In [2]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.11.0+cu126
True


In [3]:
#Load the denoiser model 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
denoiser = DenoisingAE().to(device)
denoiser.load_state_dict(torch.load("hgd_denoiser_cityscapes_best.pth", map_location=device))
denoiser.eval()

DenoisingAE(
  (enc1): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): LeakyReLU(negative_slope=0.2, inplace=True)
    (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): LeakyReLU(negative_slope=0.2, inplace=True)
  )
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (enc2): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): LeakyReLU(negative_slope=0.2, inplace=True)
    (3): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): LeakyReLU(negative_slope=

In [ ]:
#Load the object detector model 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
detector = AdversarialObjectDetection()


import glob, random
random.seed(42)

by_city = {}
for p in sorted(glob.glob("..\\leftImg8bit\\test\\*\\*.png")):
    city = p.split("\\")[-2]
    by_city.setdefault(city, []).append(p)

per_city = 150  
test_images = []
for city, paths in by_city.items():
    k = min(per_city, len(paths))
    test_images.extend(random.sample(paths, k))

test_images = sorted(test_images)
print(f"Found {len(test_images)} test images across {len(by_city)} cities")
print(f"Found {len(test_images)} test images")


TARGET_CLASS = 20
noise_configs = [
    {'type': 'gaussian', 'mean': 0, 'std': 5},
    {'type': 'salt_and_pepper', 'density': 0.05},
    {'type': 'speckle', 'intensity': 0.2},
    {'type': 'poisson', 'scale': 1.0}
]

iou_types = ["standard", "giou"]

z:\Projects\Object-Detection\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
z:\Projects\Object-Detection\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model
Found 704 test images across 6 cities
Found 704 test images


In [ ]:
#Loops through the iamges and collect IoU scores based on the parameters
def batch_test_with_loops(detector, image_paths, target_class, noise_configs,
                          iou_types, num_iterations=3, confidence_threshold=0.5,
                          denoiser=None, pipelines=None):
    all_results = []
    total_tests = len(image_paths)
    test_count = 0
    
    for image_path in image_paths:
        image_name = Path(image_path).name
        test_count += 1
        
        if test_count % 200 == 0:
            print(f"Processing image {test_count}/{total_tests}: {image_name}")
        
        try:
            original_image, adversarial_image, _ = detector.generate_adversarial_patch(
                image_path=image_path,
                target_class=target_class,
                num_iterations=num_iterations,
            )
            baseline_clean = detector.detect_objects(original_image)
            baseline_adversarial = detector.detect_objects(adversarial_image)
            
            precomputed_data = {
                'original_image': original_image,
                'adversarial_image': adversarial_image,
                'baseline_clean': baseline_clean,
                'baseline_adversarial': baseline_adversarial
            }
        except Exception as e:
            print(f"  ERROR generating adversarial image for {image_name}: {e}")
            for iou_type in iou_types:
                all_results.append({'image': image_name, 'iou_type': iou_type, 'error': str(e)})
            continue
        
        try:
            # Call to detector and IoU calculation in one step 
            results_dict = test_noise_defense_with_iou(
                detector=detector,
                image_path=image_path, 
                target_class=target_class,
                noise_configs=noise_configs,
                num_iterations=num_iterations,
                iou_types=iou_types,
                confidence_threshold=confidence_threshold,
                denoiser=denoiser,
                pipelines=pipelines,
                precomputed_data=precomputed_data)
            
            # Map resutls back into a dictionary format
            for iou_type, res in results_dict.items():
                res['image'] = image_name
                res['iou_type'] = iou_type
                all_results.append(res)
                
        except Exception as e:
            print(f"  ERROR testing noise on {image_name}: {e}")
            for iou_type in iou_types:
                all_results.append({'image': image_name, 'iou_type': iou_type, 'error': str(e)})
    
    return all_results

In [ ]:
# calculate recall and precision metrics, with optional IoU margin
def calculate_recall_precision(all_results, verbose=True, iou_margin=0.0):
    """
    Calculate Recall and Precision with optional IoU margin for match tolerance.

    If iou_margin > 0, thresholds are relaxed by that amount.
    Example: iou_margin=0.05 means Recall@0.5 uses IoU >= 0.45.
    """
    import numpy as np

    # Relax thresholds by margin, clamped to valid [0, 1] range.
    t50 = float(np.clip(0.5 - iou_margin, 0.0, 1.0))
    threshold_grid = np.clip(np.arange(0.5, 1.0, 0.05) - iou_margin, 0.0, 1.0)

    metrics = {}

    for result in all_results:
        if 'error' in result or not result.get('noise_tests'):
            continue
        iou_type = result.get('iou_type', 'standard')
        if iou_type not in metrics:
            metrics[iou_type] = {}

        seen_denoise = False
        for test in result['noise_tests']:
            cfg = test.get('noise_config', {})
            pipeline = cfg.get('pipeline', 'noise_only')
            if pipeline == 'denoise_only':
                if seen_denoise:
                    continue
                seen_denoise = True
                label = 'denoise_only'
            else:
                nt = cfg.get('type', 'unknown')
                if nt == 'gaussian':
                    base = f"gaussian_var{int(cfg['std']**2)}"
                elif nt == 'salt_and_pepper':
                    base = f"salt_pepper_{cfg['density']}"
                elif nt == 'speckle':
                    base = f"speckle_{cfg['intensity']}"
                elif nt == 'poisson':
                    base = f"poisson_{cfg['scale']}"
                else:
                    base = str(nt)
                label = f"{base}__{pipeline}"

            if label not in metrics[iou_type]:
                metrics[iou_type][label] = {
                    'num_clean_gt': 0,
                    'num_adv_preds': 0,
                    'adv_iou_vals': []
                }

            adv_rec_comp = test.get('adv_recovery_comparison', test['adversarial_comparison'])
            num_gt = adv_rec_comp.get('num_detections_1', 0)
            num_preds = adv_rec_comp.get('num_detections_2', 0)

            metrics[iou_type][label]['num_clean_gt'] += num_gt
            metrics[iou_type][label]['num_adv_preds'] += num_preds

            if 'matched_iou_values' in adv_rec_comp:
                metrics[iou_type][label]['adv_iou_vals'].extend(adv_rec_comp['matched_iou_values'])

    summary = {}
    for iou_type, data in metrics.items():
        summary[iou_type] = {}
        for label, stats in data.items():
            num_gt = stats['num_clean_gt']
            num_preds = stats['num_adv_preds']
            adv_ious = np.array(stats['adv_iou_vals'], dtype=float)

            if num_gt > 0:
                tp_50 = np.sum(adv_ious >= t50)
                recall_50 = tp_50 / num_gt
                recall_50_95 = sum(np.sum(adv_ious >= t) / num_gt for t in threshold_grid) / len(threshold_grid)
            else:
                recall_50 = np.nan
                recall_50_95 = np.nan

            #Precision checking that there are predictions, that arent all zero 
            if num_preds > 0:
                tp_50 = np.sum(adv_ious >= t50)
                precision_50 = tp_50 / num_preds
                precision_50_95 = sum(np.sum(adv_ious >= t) / num_preds for t in threshold_grid) / len(threshold_grid)
            else:
                precision_50 = np.nan
                precision_50_95 = np.nan

            #Srote the results in a dictionary
            summary[iou_type][label] = {
                'num_clean_gt': num_gt,
                'num_adv_preds': num_preds,
                'recall_50': recall_50,
                'precision_50': precision_50,
                'recall_50_95': recall_50_95,
                'precision_50_95': precision_50_95,
            }

    #prints the results
    if verbose:
        print("\n" + "=" * 100)
        print("RECALL AND PRECISION METRICS (with optional IoU margin)")
        print(f"Base threshold: 0.50 | Margin: {iou_margin:.3f} | Effective threshold: {t50:.3f}")
        print("=" * 100)
        for iou_type, data in summary.items():
            print(f"\n{'=' * 100}")
            print(f"IoU Type: {iou_type.upper()}")
            print(f"{'=' * 100}")
            print(f"{'Config':<35} | {'Recall@0.5':>10} | {'Precision@0.5':>12} | {'Recall@0.5:0.95':>15} | {'Precision@0.5:0.95':>17}")
            print("-" * 100)
            for label, stats in data.items():
                print(
                    f"{label:<35} | "
                    f"{stats['recall_50']:>10.4f} | "
                    f"{stats['precision_50']:>12.4f} | "
                    f"{stats['recall_50_95']:>15.4f} | "
                    f"{stats['precision_50_95']:>17.4f}"
                )
            print("\n")

    return summary

In [ ]:
import math

iou_types = ["standard", "giou", "diou", "ciou"]
NUM_ITERATIONS = 5
CONFIDENCE_THRESHOLD = 0.5
PIPELINES = ["noise_only", "denoise_only", "noise_then_denoise"]
RUN_COUNT = 10
IOU_MARGIN = 0.05  


all_noise_configs = [
    # Poisson
    {'type': 'poisson', 'scale': 0.1},
    {'type': 'poisson', 'scale': 0.3},
    {'type': 'poisson', 'scale': 0.9},

    # Gaussian
    {'type': 'gaussian', 'mean': 0, 'std': math.sqrt(20)},
    {'type': 'gaussian', 'mean': 0, 'std': math.sqrt(30)},
    {'type': 'gaussian', 'mean': 0, 'std': math.sqrt(40)},

    # Salt and Pepper
    {'type': 'salt_and_pepper', 'density': 0.01},
    {'type': 'salt_and_pepper', 'density': 0.03},
    {'type': 'salt_and_pepper', 'density': 0.06},

    # Speckle
    {'type': 'speckle', 'intensity': 0.05},
    {'type': 'speckle', 'intensity': 0.1},
    {'type': 'speckle', 'intensity': 0.4},
]

#Run single evaluation
def run_single_evaluation():
    x_all = batch_test_with_loops(
        detector=detector,
        image_paths=test_images,
        target_class=TARGET_CLASS,
        noise_configs=all_noise_configs,
        iou_types=iou_types,
        num_iterations=NUM_ITERATIONS,
        confidence_threshold=CONFIDENCE_THRESHOLD,
        denoiser=denoiser,
        pipelines=PIPELINES,
    )
    return calculate_recall_precision(x_all, verbose=False, iou_margin=IOU_MARGIN)

#Print Summary of the results across the runs and calculate mean and std
def summarize_run_summaries(run_summaries):
    metric_names = ['recall_50', 'precision_50', 'recall_50_95', 'precision_50_95']
    aggregated = {}

    for run_summary in run_summaries:
        for iou_type, data in run_summary.items():
            if iou_type not in aggregated:
                aggregated[iou_type] = {}
            for label, stats in data.items():
                if label not in aggregated[iou_type]:
                    aggregated[iou_type][label] = {name: [] for name in metric_names}
                for name in metric_names:
                    aggregated[iou_type][label][name].append(stats[name])

    print("\n" + "=" * 120)
    print(f"MEAN ± STD OVER {len(run_summaries)} RUNS")
    print(f"IoU margin in this run: {IOU_MARGIN:.3f}")
    print("=" * 120)

    for iou_type, data in aggregated.items():
        print(f"\n{'=' * 120}")
        print(f"IoU Type: {iou_type.upper()}")
        print(f"{'=' * 120}")
        print(f"{'Config':<35} | {'Recall@0.5':>19} | {'Precision@0.5':>19} | {'Recall@0.5:0.95':>19} | {'Precision@0.5:0.95':>19}")
        print("-" * 120)

        for label, stats in data.items():
            def mean_std(values):
                values = np.array(values, dtype=float)
                mean_value = np.nanmean(values)
                std_value = np.nanstd(values, ddof=1) if np.sum(~np.isnan(values)) > 1 else 0.0
                return mean_value, std_value

            r50_mean, r50_std = mean_std(stats['recall_50'])
            p50_mean, p50_std = mean_std(stats['precision_50'])
            r5095_mean, r5095_std = mean_std(stats['recall_50_95'])
            p5095_mean, p5095_std = mean_std(stats['precision_50_95'])

            print(
                f"{label:<35} | "
                f"{r50_mean:>8.4f} ± {r50_std:>8.4f} | "
                f"{p50_mean:>8.4f} ± {p50_std:>8.4f} | "
                f"{r5095_mean:>8.4f} ± {r5095_std:>8.4f} | "
                f"{p5095_mean:>8.4f} ± {p5095_std:>8.4f}"
            )
        print("\n")

    return aggregated


run_summaries = []
for _ in range(RUN_COUNT):
    run_summaries.append(run_single_evaluation())

summarize_run_summaries(run_summaries)

Processing image 200/704: bielefeld_000000_021625_leftImg8bit.png
Processing image 400/704: leverkusen_000053_000019_leftImg8bit.png
Processing image 600/704: munich_000126_000019_leftImg8bit.png
Processing image 200/704: bielefeld_000000_021625_leftImg8bit.png
Processing image 400/704: leverkusen_000053_000019_leftImg8bit.png
Processing image 600/704: munich_000126_000019_leftImg8bit.png
Processing image 200/704: bielefeld_000000_021625_leftImg8bit.png
Processing image 400/704: leverkusen_000053_000019_leftImg8bit.png
Processing image 600/704: munich_000126_000019_leftImg8bit.png
Processing image 200/704: bielefeld_000000_021625_leftImg8bit.png
Processing image 400/704: leverkusen_000053_000019_leftImg8bit.png
Processing image 600/704: munich_000126_000019_leftImg8bit.png
Processing image 200/704: bielefeld_000000_021625_leftImg8bit.png
Processing image 400/704: leverkusen_000053_000019_leftImg8bit.png
Processing image 600/704: munich_000126_000019_leftImg8bit.png
Processing image 200

{'standard': {'poisson_0.1__noise_only': {'recall_50': [np.float64(0.521683400982696),
    np.float64(0.5142063661610767),
    np.float64(0.509720145268105),
    np.float64(0.5187994018372143),
    np.float64(0.5081179235206152),
    np.float64(0.5165562913907285),
    np.float64(0.5123905148472548),
    np.float64(0.5184789574877163),
    np.float64(0.5276650288399914),
    np.float64(0.5099337748344371)],
   'precision_50': [np.float64(0.7341049150759056),
    np.float64(0.735748127770136),
    np.float64(0.7264423808798904),
    np.float64(0.7363553668890237),
    np.float64(0.7273700305810398),
    np.float64(0.7370827617741198),
    np.float64(0.7298037425832953),
    np.float64(0.7352317479551651),
    np.float64(0.7424105801021942),
    np.float64(0.7267468412239306)],
   'recall_50_95': [np.float64(0.39164708395641956),
    np.float64(0.3853556932279427),
    np.float64(0.3821832941679128),
    np.float64(0.3886989959410383),
    np.float64(0.37950224311044645),
    np.float64(